<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎬 LTX-2.3 22B Distilled (quanto int8)</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Kaggle P100 GPU Edition — Created by <strong>AIQUEST</strong></h3>
  <p style='color: #ddd; margin: 0 0 20px 0;'>Free on Kaggle P100 GPU | Wan2GP Engine + mmgp Profile 4</p>
</div>

---

<div align="center">

  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Kaggle-Free%20T4%20GPU-20BEFF?style=for-the-badge&logo=kaggle&logoColor=white" />

 <br>

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/▶%20Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20𝕏-0000?style=for-the-badge&logo=x&logoColor=white" />
  </a>

</div>

---

### What is LTX-2.3 22B Distilled?

**LTX-2.3 22B Distilled** is a powerful video generation model running on free Kaggle T4 GPU via the **Wan2GP** engine with intelligent GPU offloading.

| Feature | Details |
|----|----|
| Model | LTX-2.3 22B Distilled (quanto int8) |
| Engine | Wan2GP + mmgp Profile 4 |
| Pipeline | Two-stage: 8 steps (half-res) → 2x upscale → 3 steps (refine) |
| VRAM | Fits in 15GB P100 VRAM |
| RAM | 29GB Kaggle RAM — no swap needed |
| Modes | Text-to-Video, Image-to-Video |

### Quick Start
1. **Settings → Accelerator → GPU P100 x1**
2. **Turn on Internet** in Settings sidebar
3. Run all cells in order
4. Use the Gradio UI link to generate



---
## Step 1: Environment Setup

Optimizes memory settings for Kaggle P100 GPU.

> If no GPU is detected → **Settings → Accelerator → GPU P100 x1**

In [1]:
# Cell 1: Dual T4 Environment Optimization
import os, gc, psutil, torch

print("=== Kaggle Dual T4 Environment Setup ===")

# 1. Clear System Caches
os.system("echo 3 | sudo tee /proc/sys/vm/drop_caches > /dev/null 2>&1")
os.system("echo 1 | sudo tee /proc/sys/vm/overcommit_memory > /dev/null 2>&1")

# 2. CUDA Memory Management for Dual GPUs
# We set max_split_size_mb to prevent one large allocation from blocking 
# the VRAM needed for cross-GPU transfers.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128,garbage_collection_threshold:0.6"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID" # Ensures GPU 0 and 1 don't swap IDs
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# 3. Validation
gpu_count = torch.cuda.device_count()
print(f"GPUs Detected: {gpu_count}")
for i in range(gpu_count):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB)")

ram = psutil.virtual_memory()
print(f"System RAM: {ram.total / 1024**3:.1f} GB total")

gc.collect()
torch.cuda.empty_cache()

if gpu_count < 2:
    print("\n⚠️ WARNING: Only 1 GPU detected. Go to Settings -> Accelerator -> GPU T4 x2")
else:
    print("\n✅ Dual GPU Environment Optimized!")

=== Kaggle Dual T4 Environment Setup ===
GPUs Detected: 2
  GPU 0: Tesla T4 (14.6 GB)
  GPU 1: Tesla T4 (14.6 GB)
System RAM: 31.4 GB total

✅ Dual GPU Environment Optimized!


---
## Step 2: Clone Wan2GP & Install Dependencies

Clones the Wan2GP repository and installs all required packages.

In [2]:
# Cell 1: Clone Wan2GP & install dependencies
import subprocess
try:
    subprocess.run(["nvidia-smi"], check=True)
    print("GPU Active!")
except Exception:
    print("WARNING: No GPU. Go to Settings → Accelerator → GPU T4 x1")

!git clone https://github.com/DeepBeepMeep/Wan2GP.git
!pip install --timeout 120 --retries 5 -q -r Wan2GP/requirements.txt
!pip install --timeout 120 --retries 5 -q mmgp gradio

Sat Apr 18 14:46:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

---
## Step 3: Download All Required Models

Downloads ~48GB of model files. Large files go to `/kaggle/tmp` and are symlinked back to save working disk space.

In [3]:
# Cell 2: Download all required models (Kaggle disk-aware)
import os
from huggingface_hub import hf_hub_download

REPO = "DeepBeepMeep/LTX-2"
MODEL_DIR = "Wan2GP/models"
TMP_DIR = "/kaggle/tmp/models"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)

# === Large files go to /kaggle/tmp, symlinked back ===
LARGE_FILES = [
    "ltx-2.3-22b-distilled_diffusion_model_quanto_int8.safetensors",  # 19.4 GB
    "ltx-2.3-22b-distilled-lora-384.safetensors",                      # 7.6 GB
    "ltx-2.3-22b_embeddings_connector.safetensors",                     # 4.0 GB
    "ltx-2.3-22b_text_embedding_projection.safetensors",                # 2.3 GB
    "ltx-2.3-22b_vae.safetensors",                                      # 1.5 GB
]

for f in LARGE_FILES:
    dest = os.path.join(MODEL_DIR, f)
    if os.path.exists(dest):
        print(f"  ✓ Already exists: {f}")
        continue
    print(f"Downloading {f} → /kaggle/tmp ...")
    hf_hub_download(repo_id=REPO, filename=f, local_dir=TMP_DIR)
    actual = os.path.join(TMP_DIR, f)
    os.symlink(actual, dest)
    print(f"  ✓ {f} (symlinked)")

# === Small files download normally to /kaggle/working ===
SMALL_FILES = [
    "ltx-2.3-22b_audio_vae.safetensors",
    "ltx-2.3-22b_vocoder.safetensors",
    "ltx-2.3-spatial-upscaler-x2-1.1.safetensors",
    "ltx-2.3-temporal-upscaler-x2-1.0.safetensors",
]

for f in SMALL_FILES:
    dest = os.path.join(MODEL_DIR, f)
    if os.path.exists(dest):
        print(f"  ✓ Already exists: {f}")
        continue
    print(f"Downloading {f}...")
    hf_hub_download(repo_id=REPO, filename=f, local_dir=MODEL_DIR)
    print(f"  ✓ {f}")

# === Gemma text encoder — large, goes to /kaggle/tmp ===
GEMMA_FOLDER = "gemma-3-12b-it-qat-q4_0-unquantized"
GEMMA_FILES = [
    "gemma-3-12b-it-qat-q4_0-unquantized_quanto_bf16_int8.safetensors",
    "added_tokens.json",
    "chat_template.json",
    "config_light.json",
    "generation_config.json",
    "preprocessor_config.json",
    "processor_config.json",
    "special_tokens_map.json",
    "tokenizer.json",
    "tokenizer.model",
    "tokenizer_config.json",
]

# Download gemma to /kaggle/tmp, symlink the whole folder
gemma_dest = os.path.join(MODEL_DIR, GEMMA_FOLDER)
gemma_tmp = os.path.join(TMP_DIR, GEMMA_FOLDER)

if os.path.exists(gemma_dest):
    print(f"  ✓ Already exists: {GEMMA_FOLDER}/")
else:
    os.makedirs(gemma_tmp, exist_ok=True)
    for gf in GEMMA_FILES:
        tmp_file = os.path.join(gemma_tmp, gf)
        if os.path.exists(tmp_file):
            print(f"  ✓ Already exists: gemma/{gf}")
            continue
        print(f"Downloading gemma/{gf} → /kaggle/tmp ...")
        hf_hub_download(
            repo_id=REPO,
            filename=f"{GEMMA_FOLDER}/{gf}",
            local_dir=TMP_DIR,
        )
        print(f"  ✓ gemma/{gf}")
    # Symlink the whole gemma folder
    os.symlink(gemma_tmp, gemma_dest)
    print(f"  ✓ {GEMMA_FOLDER}/ (symlinked)")

# Clean up HF download cache to free disk
import shutil
cache_dir = os.path.join(MODEL_DIR, ".cache")
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
tmp_cache = os.path.join(TMP_DIR, ".cache")
if os.path.exists(tmp_cache):
    shutil.rmtree(tmp_cache)

os.system("df -h /kaggle/working /kaggle/tmp")
print("\n✅ All downloads complete!")

ltx-2.3-22b-distilled_diffusion_model_qu(…):   0%|          | 0.00/19.4G [00:00<?, ?B/s]

  ✓ ltx-2.3-22b-distilled_diffusion_model_quanto_int8.safetensors (symlinked)


ltx-2.3-22b-distilled-lora-384.safetenso(…):   0%|          | 0.00/7.61G [00:00<?, ?B/s]

  ✓ ltx-2.3-22b-distilled-lora-384.safetensors (symlinked)


ltx-2.3-22b_embeddings_connector.safeten(…):   0%|          | 0.00/4.03G [00:00<?, ?B/s]

...

  ✓ gemma/tokenizer.json


gemma-3-12b-it-qat-q4_0-unquantized/toke(…):   0%|          | 0.00/4.69M [00:00<?, ?B/s]

  ✓ gemma/tokenizer.model


tokenizer_config.json: 0.00B [00:00, ?B/s]

  ✓ gemma/tokenizer_config.json
  ✓ gemma-3-12b-it-qat-q4_0-unquantized/ (symlinked)
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  1.6G   18G   9% /kaggle/working
overlay         7.9T  6.8T  1.2T  86% /

✅ All downloads complete!


---
## Step 4: Write the Generation Script

Creates `run_ltx.py` with the full pipeline: model loading, mmgp offloading, generation logic, and the branded Gradio UI.

In [4]:
%%writefile run_ltx.py
import gc
import os
import sys
import random
import tempfile
import glob
import traceback
import numpy as np
import subprocess
import psutil
from PIL import Image

# ---- bootstrap Wan2GP ----
WAN2GP_DIR = os.path.abspath("Wan2GP")
sys.path.insert(0, WAN2GP_DIR)
os.chdir(WAN2GP_DIR)
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128,garbage_collection_threshold:0.5"

import torch
import torchaudio
import gradio as gr
from shared.utils.audio_video import save_video

# ==== DUAL GPU SETUP ====
device_count = torch.cuda.device_count()
primary_gpu = "cuda:0"
secondary_gpu = "cuda:1" if device_count > 1 else "cuda:0"
print(f"Detected {device_count} GPUs. Primary: {primary_gpu}, Secondary: {secondary_gpu}")

# ==== LOAD MODEL ====
from mmgp import offload
from shared.utils import files_locator as fl
fl.set_checkpoints_paths(["models", "ckpts", "."])
from models.ltx2.ltx2_handler import family_handler

base_model_type = "ltx2_22B"
model_def = {"ltx2_pipeline": "distilled"}
extra = family_handler.query_model_def(base_model_type, model_def)
model_def.update(extra)

gemma_folder = "models/gemma-3-12b-it-qat-q4_0-unquantized"
text_encoder_file = glob.glob(os.path.join(gemma_folder, "*quanto*.safetensors"))[0]
transformer_path = os.path.join("models", "ltx-2.3-22b-distilled_diffusion_model_quanto_int8.safetensors")

ltx2_model, pipe = family_handler.load_model(
    model_filename=transformer_path,
    model_type="ltx2_22B_distilled",
    base_model_type=base_model_type,
    model_def=model_def,
    dtype=torch.bfloat16,
    VAE_dtype=torch.float16,
    text_encoder_filename=text_encoder_file,
)

# ==== DUAL GPU DISTRIBUTION & PATCHING ====
def apply_device_patch(model, target_device):
    """Ensures inputs move to secondary GPU and outputs return to primary GPU."""
    try: target_dtype = next(model.parameters()).dtype
    except: target_dtype = torch.float16
    
    def make_patched_fn(orig_fn):
        def patched_fn(*args, **kwargs):
            new_args = [a.to(device=target_device, dtype=target_dtype) if isinstance(a, torch.Tensor) else a for a in args]
            new_kwargs = {k: (v.to(device=target_device, dtype=target_dtype) if isinstance(v, torch.Tensor) else v) for k, v in kwargs.items()}
            out = orig_fn(*new_args, **new_kwargs)
            if isinstance(out, torch.Tensor): return out.to(device=primary_gpu)
            if isinstance(out, (list, tuple)): return type(out)([o.to(device=primary_gpu) if isinstance(o, torch.Tensor) else o for o in out])
            return out
        return patched_fn

    for m in ["forward", "encode", "decode", "__call__"]:
        if hasattr(model, m): setattr(model, m, make_patched_fn(getattr(model, m)))

# Move VAE, Encoders, and Upscalers to GPU 1
move_keys = ["vae", "audio_vae", "audio_encoder", "audio_decoder", "vocoder", "spatial_upsampler", "temporal_upsampler", "video_encoder"]
for key in move_keys:
    if pipe.get(key) is not None:
        pipe[key].to(secondary_gpu)
        apply_device_patch(pipe[key], secondary_gpu)

# mmgp only manages huge components remaining on GPU 0
mmgp_pipe = {k: v for k, v in pipe.items() if v is not None and k not in move_keys}
offload.profile(mmgp_pipe, profile_no=4, budgets={"transformer": 8500, "text_encoder": 2000, "*": 500})
offload.shared_state["_attention"] = "sdpa"

def get_resolution(base_res_str, aspect_ratio_str):
    base_resolutions = {"1080p": 1088, "720p": 704, "540p": 544, "480p": 480}
    ratios = {"16:9 Landscape": 16/9, "4:3 Standard": 4/3, "1:1 Square": 1.0, "9:16 Portrait": 9/16}
    base = base_resolutions.get(base_res_str, 704)
    ratio = ratios.get(aspect_ratio_str, 16/9)
    if ratio >= 1.0: h, w = base, int(base * ratio)
    else: w, h = base, int(base / ratio)
    return (w // 32) * 32, (h // 32) * 32

@torch.inference_mode()
def Video_Generation(prompt, img_s_p, img_m_paths, m_indices_str, img_e_p, audio_path, seed, dur_str, res_str, asp_str, strength, progress=gr.Progress()):
    try:
        gc.collect(); torch.cuda.empty_cache()
        dur_map = {
            "2 Seconds (49 frames)": 49, "3 Seconds (73 frames)": 73, "5 Seconds (121 frames)": 121, 
            "10 Seconds (241 frames)": 241, "15 Seconds (361 frames)": 361, "20 Seconds (481 frames)": 481
        }
        num_frames = dur_map.get(dur_str, 121)
        width, height = get_resolution(res_str, asp_str)

        if seed is None or seed < 0: seed = random.randint(0, 2**32 - 1)
        
        # 1. Resize and Prepare Images
        def prep(p): return Image.open(p).convert("RGB").resize((width, height), Image.LANCZOS) if p else None
        img_s = prep(img_s_p)
        img_e = prep(img_e_p)

        # 2. Process Multiple Middle Frames and Indices
        image_refs = []
        image_indices = []
        
        if img_m_paths:
            try:
                provided_indices = [int(x.strip()) for x in m_indices_str.split(",") if x.strip()]
            except ValueError:
                return None, "❌ Error: Frame indices must be numbers separated by commas (e.g. 30, 60)."

            if len(provided_indices) != len(img_m_paths):
                return None, f"❌ Mismatch: You uploaded {len(img_m_paths)} images but provided {len(provided_indices)} indices."

            img_m_paths.sort() 
            for path, idx in zip(img_m_paths, provided_indices):
                img = prep(path)
                if img:
                    if 0 < idx < num_frames - 1:
                        image_refs.append(img)
                        image_indices.append(idx)
                    else:
                        return None, f"❌ Error: Frame index {idx} is out of bounds for a {num_frames} frame video."

        # 3. Fix Audio duration mismatch (Demon Voice Fix)
        final_audio_input = None
        if audio_path:
            wav, sr = torchaudio.load(audio_path)
            if sr != 24000: wav = torchaudio.transforms.Resample(sr, 24000)(wav)
            if wav.shape[0] > 1: wav = wav.mean(0, keepdim=True)
            target_samples = int(((num_frames - 1) / 24.0) * 24000)
            if wav.shape[1] > target_samples: wav = wav[:, :target_samples]
            else: wav = torch.nn.functional.pad(wav, (0, target_samples - wav.shape[1]))
            final_audio_input = tempfile.mktemp(suffix=".wav")
            torchaudio.save(final_audio_input, wav, 24000)

        def cb(step, latent, is_start, **kwargs):
            if not is_start: progress(step/10, desc=f"Sampling Step {step}...")

        gen_kwargs = dict(
            input_prompt=prompt,
            image_start=img_s,
            image_end=img_e,
            image_refs=image_refs if image_refs else None,
            image_indices=image_indices if image_indices else None,
            input_audio=final_audio_input, 
            height=height, width=width,
            frame_num=num_frames, fps=24.0,
            seed=int(seed), callback=cb,
            VAE_tile_size=128, enhance_prompt=True,
            input_video_strength=float(strength),
        )

        print(f"🚀 Gen Start | {width}x{height} | Indices: {image_indices} | Audio: {'Custom' if audio_path else 'Native'}")
        result = ltx2_model.generate(**gen_kwargs)
        if result is None: return None, "Generation failed."

        video_tensor = result.get("x") if isinstance(result, dict) else result
        native_audio = result.get("audio") if isinstance(result, dict) else None

        out_path = tempfile.mktemp(suffix=".mp4")
        save_video(video_tensor.unsqueeze(0).float().cpu() / 127.5 - 1.0, out_path, fps=24.0, normalize=True, value_range=(-1, 1))

        # Final Muxing
        final_mux_audio = audio_path if audio_path else None
        if not audio_path and native_audio is not None:
            final_mux_audio = tempfile.mktemp(suffix=".wav")
            import soundfile as sf
            a_np = native_audio if isinstance(native_audio, np.ndarray) else native_audio.cpu().numpy()
            if a_np.ndim == 2 and a_np.shape[0] <= 2: a_np = a_np.T
            sf.write(final_mux_audio, a_np, 24000)

        if final_mux_audio:
            final_path = out_path.replace(".mp4", "_final.mp4")
            subprocess.run(["ffmpeg", "-y", "-i", out_path, "-i", final_mux_audio, "-c:v", "copy", "-c:a", "aac", "-b:a", "192k", "-shortest", final_path], check=True, capture_output=True)
            out_path = final_path

        return out_path, f"✅ Success! Injected Indices: {image_indices}"

    except Exception as e:
        traceback.print_exc()
        return None, f"❌ Error: {str(e)}"

# ==== GRADIO UI ====
CSS = ".gradio-container { max-width: 1100px !important; margin: auto !important; } .brand-header { text-align: center; background: linear-gradient(135deg, #1e3a8a 0%, #4c1d95 100%); padding: 20px; border-radius: 12px; color: white; margin-bottom: 20px; }"
with gr.Blocks(css=CSS, theme=gr.themes.Soft()) as demo:
    gr.HTML('<div class="brand-header"><h1>🎬 LTX-2.3 Multi-Guidance Studio</h1><p>Keyframe Control & Aligned Audio Sync</p></div>')
    with gr.Row():
        with gr.Column(scale=4):
            prompt = gr.Textbox(label="🎬 Scene Description", lines=4)
            audio = gr.Audio(label="🎵 Audio Track (Sync Guidance)", type="filepath")
            with gr.Row():
                with gr.Column():
                    img_s = gr.Image(type="filepath", label="Start Frame (Frame 0)")
                with gr.Column():
                    img_m = gr.File(file_count="multiple", file_types=["image"], label="Middle Image(s)")
                    m_indices = gr.Textbox(label="Frame Indices", placeholder="e.g. 40, 80", info="Indices for uploaded middle images")
                with gr.Column():
                    img_e = gr.Image(type="filepath", label="Last Frame (End)")
        with gr.Column(scale=2):
            seed = gr.Number(label="🎲 Seed", value=-1, precision=0)
            strength = gr.Slider(label="💪 Guidance Strength", minimum=0.0, maximum=1.0, value=0.75, step=0.05)
            dur = gr.Dropdown(label="⏱️ Video Length", choices=["2 Seconds (49 frames)", "3 Seconds (73 frames)", "5 Seconds (121 frames)", "10 Seconds (241 frames)", "15 Seconds (361 frames)", "20 Seconds (481 frames)"], value="5 Seconds (121 frames)")
            res = gr.Dropdown(label="📐 Resolution", choices=["1080p", "720p", "540p", "480p"], value="720p")
            asp = gr.Dropdown(label="📏 Aspect Ratio", choices=["16:9 Landscape", "4:3 Standard", "1:1 Square", "9:16 Portrait"], value="16:9 Landscape")
            gen_btn = gr.Button("🚀 Generate Video", variant="primary")
    video_out = gr.Video(label="🎥 Result")
    status_out = gr.Textbox(label="ℹ️ Status", interactive=False)
    
    gen_btn.click(
        fn=Video_Generation, 
        inputs=[prompt, img_s, img_m, m_indices, img_e, audio, seed, dur, res, asp, strength], 
        outputs=[video_out, status_out]
    )

demo.queue().launch(share=True, inline=False, debug=True)

Writing run_ltx.py


---
## Step 5: Launch! 🚀

Runs the generation script. Watch for the **public Gradio URL** in the output.

In [7]:
# Cell 4: Launch!
!cd /kaggle/working && python -u run_ltx.py 2>&1

Detected 2 GPUs. Primary: cuda:0, Secondary: cuda:1
2026-04-18 17:26:14.462101: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776533174.498096     924 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776533174.507903     924 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776533174.556905     924 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776533174.556944     924 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776533174.556948     9

---

<div align="center">

### 🎉 Enjoyed this notebook?

If this was helpful, please **⬆️ upvote** and **subscribe** for more free AI tools & tutorials!

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/▶%20Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20𝕏-0000?style=for-the-badge&logo=x&logoColor=white" />
  </a>

**Made with ❤️ by AIQUEST** | [@aiquestacademy](https://youtube.com/@aiquestacademy)

</div>